In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.utils import to_categorical

# load ascii text and covert to lowercase

In [ ]:
Text_data= "/content/output.txt"
# Try a different encoding like 'latin-1'
raw_text = open(Text_data, 'r', encoding='latin-1').read()
raw_text = raw_text.lower()
print(raw_text )

In [ ]:
#Step-1: Unique characters nikalna aur unhe sort karna
chars = sorted(list(set(raw_text)))
print("Sorted unique characters:")
print(chars)

In [ ]:
#Step-2: Har unique character ko ek integer se map karna
char_integer = {c: i for i, c in enumerate(chars)}
print("Mapping of unique characters to integer indices:")
print(char_integer)

In [ ]:
#Step-3: Calculate distance from original positions
original_positions = {c: raw_text.index(c) for c in chars}
distances = {c: abs(original_positions[c] - char_integer[c]) for c in chars}
print("Distance between original positions and mapped indices:")
print(distances)

In [ ]:
#Step-4: Har unique character ki occurrences count karna
occurrences = {c: raw_text.count(c) for c in chars}
print("Occurrences of each unique character:")
print(occurrences)

In [ ]:
#Step-5: Har unique character ka pehla aur aakhri position nikalna
positions = {c: (raw_text.index(c), raw_text.rindex(c)) for c in chars}
print("First and last positions of each unique character:")
print(positions)

In [ ]:
#Step-6: Har unique character ka frequency percentage nikalna
total_chars = len(raw_text)
frequency_percentage = {c: (occurrences[c] / total_chars) * 100 for c in chars}
print("Frequency percentage of each unique character:")
print(frequency_percentage)

In [ ]:
!pip install langdetect

In [ ]:
#Step-7:Language detection karna
from langdetect import detect

language = detect(raw_text)
print("Detected language of the string:", language)

In [ ]:
#Step-8: Sab se zyada any wala character our  Sab se kam any wala character:
most_frequent_char = max(occurrences, key=occurrences.get)

least_frequent_char = min(occurrences, key=occurrences.get)

print(f"The most frequent character is '{most_frequent_char}' with {occurrences[most_frequent_char]} occurrences.")
print(f"The least frequent character is '{least_frequent_char}' with {occurrences[least_frequent_char]} occurrences.")

In [ ]:
#Step-9: Character distribution visualize karna
# Define figure size (width, height) in inches
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the bar chart
ax.bar(frequency_percentage.keys(), frequency_percentage.values())

# Set labels and title
ax.set_xlabel('Characters')
ax.set_ylabel('Frequency Percentage')
ax.set_title('Character Frequency Distribution')

# Display the plot
plt.show()

In [ ]:
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab: ", n_vocab)

In [ ]:
# Given parameters and data
seq_length = 80
n_chars = len(raw_text)
# Initialize empty lists for input-output pairs
dataX = []
dataY = []

# Loop to create sequences and their corresponding outputs
for i in range(0, n_chars - seq_length, 1):
    seq_in = raw_text[i:i + seq_length]  # Input sequence of length seq_length
    seq_out = raw_text[i + seq_length]   # Output character following the input sequence

    # Convert characters to integers using char_integer mapping
    seq_in_int = [char_integer[char] for char in seq_in]
    seq_out_int = char_integer[seq_out]

    # Append the integer sequences to dataX and dataY
    dataX.append(seq_in_int)
    dataY.append(seq_out_int)

# Calculate the total number of patterns
n_patterns = len(dataX)

# Print the total number of patterns
print("Total Patterns:", n_patterns)


In [ ]:
print(dataX)

In [ ]:
print(dataY)

In [ ]:
# reshape X to be [samples, time steps, features]
X = np.reshape(dataX, (n_patterns, seq_length, 1))
print("Reshaped shape:", X.shape)
# normalize
x = X / float(n_vocab)
print("Normalize shape:",x.shape)
# one hot encode the output variable
y = to_categorical(dataY)
print("One-hot encoded shape:", y.shape)

In [ ]:
print(x.shape[1])
print(x.shape[2])

# **Build LSTM Model**

In [ ]:
model = Sequential()
model.add(LSTM(256, input_shape=(x.shape[1], x.shape[2])))  #input_shape=(time_steps, features)
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

In [ ]:
model.summary()

In [ ]:
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# **Define the checkpoint**

In [ ]:
filepath = "weights-improvement-{epoch:02d}-{loss:.4f}.hdf5"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

# **Fit Model**

In [ ]:
model.fit(x, y, epochs=25 , batch_size= 100, callbacks=[callbacks_list,EarlyStopping(monitor = 'loss', patience=3,verbose=1)])

weights-improvement-01-3.0302.hdf5
weights-improvement-11-2.3848.hdf5

#**load the network weights**

In [ ]:
filename = "weights-improvement-05-2.9678.hdf5"
model.load_weights(filename)
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# **pick a random seed**

In [ ]:
start = np.random.randint(0, len(dataX)-1)
pattern = dataX[start]
int_to_char = dict((i, c) for i, c in enumerate(chars))
print("Random Seed:")
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")

In [ ]:
generated_text = []
for i in range(1000):
 x = np.reshape(pattern, (1, len(pattern), 1))
 x = x / float(n_vocab)
 prediction = model.predict(x, verbose=0)
 index = np.argmax(prediction)
 result = int_to_char[index]
 seq_in = [int_to_char[value] for value in pattern]
 sys.stdout.write(result)
 pattern.append(index)
 pattern = pattern[1:len(pattern)]
print("\nGenerated Text:")
print(''.join(generated_text))